# E-commerce Revenue & Customer Intelligence

A reproducible descriptive analysis of UCI Online Retail II. Questions: when does the retailer sell most, how do returns affect line sales, and how do repeat purchasing and customer concentration vary?

> Revenue is calculated as quantity × unit price. This is line-level sales value, not accounting revenue or profit. Returns, missing IDs, non-merchandise codes, and wholesale outliers are handled explicitly as limitations.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('../data/online_retail_II.xlsx')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/online_retail_II.xlsx')
if not DATA_PATH.exists():
    raise FileNotFoundError('Put the official UCI workbook at data/online_retail_II.xlsx')

sheets = pd.read_excel(DATA_PATH, sheet_name=None)
[(name, sheet.shape) for name, sheet in sheets.items()]


## Prepare the transaction rows

The workbook sheets overlap. Keep the maximum multiplicity of every exact row found in either sheet. This removes the cross-sheet overlap while preserving repeated lines within a sheet. Hashes are used as a practical row key.


In [ ]:
frames = list(sheets.values())
columns = frames[0].columns.tolist()
hashes_by_sheet = [
    pd.util.hash_pandas_object(frame[columns], index=False).astype('uint64').value_counts()
    for frame in frames
]
max_repeats = pd.concat(hashes_by_sheet, axis=1).fillna(0).max(axis=1)
retail = pd.concat(frames, ignore_index=True)
retail['_row_hash'] = pd.util.hash_pandas_object(retail[columns], index=False).astype('uint64')
retail['_occurrence'] = retail.groupby('_row_hash', sort=False).cumcount()
keep = retail['_occurrence'] < retail['_row_hash'].map(max_repeats)
removed_cross_sheet = int((~keep).sum())
retail = retail.loc[keep, columns].copy()
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'], errors='coerce')
retail['Invoice'] = retail['Invoice'].astype('string')
retail['IsCancellation'] = retail['Invoice'].str.startswith('C', na=False)
retail['LineRevenue'] = retail['Quantity'] * retail['Price']
print(f'Removed excess cross-sheet copies: {removed_cross_sheet:,}')
print(f'Retained transaction rows: {len(retail):,}')
retail.head()


## Data quality and cancellations

Rows with a missing customer ID remain usable for product and country totals, but cannot be assigned to customer cohorts or RFM segments. The UCI workbook includes returns and cancellation records; compare the cancellation invoice prefix with negative quantities before defining net sales.


In [ ]:
quality = pd.DataFrame({
    'metric': ['Rows', 'Missing customer IDs', 'Missing descriptions', 'Exact within-sheet duplicates (for context)', 'Negative quantity lines', 'C-prefixed invoice lines'],
    'value': [len(retail), retail['Customer ID'].isna().sum(), retail['Description'].isna().sum(), sum(int(s.duplicated().sum()) for s in frames), (retail['Quantity'] < 0).sum(), retail['IsCancellation'].sum()]
})
quality


In [ ]:
retail['NegativeQuantity'] = retail['Quantity'] < 0
cancellation_check = pd.crosstab(retail['IsCancellation'], retail['NegativeQuantity'], margins=True)
cancellation_check


## Gross sales and returns over time

Positive quantity/price lines are counted as gross sales. Negative quantity/positive price lines are shown as returns. The net line-sales measure subtracts their absolute value; it is not a formal accounting measure.


In [ ]:
positive = retail.loc[(retail['Quantity'] > 0) & (retail['Price'] > 0)].copy()
negative = retail.loc[(retail['Quantity'] < 0) & (retail['Price'] > 0)].copy()
positive['Month'] = positive['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
negative['Month'] = negative['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
gross = positive.groupby('Month')['LineRevenue'].sum().rename('Gross sales')
returns = (-negative.groupby('Month')['LineRevenue'].sum()).rename('Returns')
monthly = pd.concat([gross, returns], axis=1).fillna(0)
monthly['Net line sales'] = monthly['Gross sales'] - monthly['Returns']
print(f"Gross positive line sales: GBP {positive['LineRevenue'].sum():,.2f}")
print(f"Negative-quantity line value: GBP {-negative['LineRevenue'].sum():,.2f}")
print(f"Mechanical net line sales: GBP {positive['LineRevenue'].sum() + negative['LineRevenue'].sum():,.2f}")
monthly.tail()


In [ ]:
sns.set_theme(style='whitegrid')
ax = monthly.plot(figsize=(11, 5), marker='o')
ax.set(title='Monthly sales and returns', xlabel='Month', ylabel='GBP')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Customer cohorts

A cohort is grouped by the month of the customer's first positive purchase. Retention is the share of that cohort placing at least one order in each later month. Customer IDs are missing for some transactions, so this is based only on identified buyers.


In [ ]:
identified = positive.dropna(subset=['Customer ID']).copy()
identified['InvoiceMonth'] = identified['InvoiceDate'].dt.to_period('M')
customer_month = identified.groupby(['Customer ID', 'InvoiceMonth'])['Invoice'].nunique().rename('Orders').reset_index()
first_month = customer_month.groupby('Customer ID')['InvoiceMonth'].min().rename('CohortMonth')
customer_month = customer_month.join(first_month, on='Customer ID')
customer_month['CohortIndex'] = (customer_month['InvoiceMonth'].dt.year - customer_month['CohortMonth'].dt.year) * 12 + customer_month['InvoiceMonth'].dt.month - customer_month['CohortMonth'].dt.month
cohort_counts = customer_month.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().unstack(fill_value=0)
retention = cohort_counts.divide(cohort_counts.iloc[:, 0], axis=0)
month_one = retention.iloc[:, 1].dropna()
print(f'Median month-one retention: {month_one.median():.1%} across {len(month_one)} eligible cohorts')


In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(retention, mask=retention.isna(), cmap='Blues', vmin=0, vmax=1, annot=True, fmt='.0%', linewidths=.3, ax=ax)
ax.set(title='Customer retention by first-purchase cohort', xlabel='Months since first purchase', ylabel='Cohort month')
plt.tight_layout()
plt.show()


## Market mix and product caveats

The raw stock-code list includes postage and manual adjustments, so rankings below are transaction-code rankings rather than a clean merchandise catalog. Review special codes and exceptional bulk orders before drawing product recommendations.


In [ ]:
positive['RevenueGBP'] = positive['Quantity'] * positive['Price']
country = positive.groupby('Country').agg(RevenueGBP=('RevenueGBP', 'sum'), Invoices=('Invoice', 'nunique')).sort_values('RevenueGBP', ascending=False)
print(f"Leading country share: {country.iloc[0].name} — {country.iloc[0].RevenueGBP / country.RevenueGBP.sum():.1%}")
country.head(10)


In [ ]:
products = positive.groupby(['StockCode', 'Description'], dropna=False).agg(RevenueGBP=('RevenueGBP', 'sum'), Units=('Quantity', 'sum'), Invoices=('Invoice', 'nunique')).sort_values('RevenueGBP', ascending=False)
products.head(15)


## Exploratory customer segments (RFM)

Recency, frequency, and monetary values are ranked into dataset-relative quartiles. Segment labels are heuristics for exploration, not predictions of future value.


In [ ]:
snapshot = positive['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
rfm = identified.groupby('Customer ID').agg(
    RecencyDays=('InvoiceDate', lambda s: (snapshot - s.max().normalize()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('LineRevenue', 'sum'),
)
rfm['R'] = pd.qcut(rfm['RecencyDays'].rank(method='first', ascending=False), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFMScore'] = rfm[['R', 'F', 'M']].sum(axis=1)
def assign_segment(row):
    if row.RFMScore >= 10: return 'Champions'
    if row.R >= 3 and row.F >= 3: return 'Loyal'
    if row.R >= 3 and row.F <= 2: return 'New / potential'
    if row.R <= 2 and row.F >= 3: return 'At risk'
    return 'Needs attention'
rfm['Segment'] = rfm.apply(assign_segment, axis=1)
segment_summary = rfm.groupby('Segment').agg(Customers=('RecencyDays', 'size'), RevenueGBP=('Monetary', 'sum'), MedianFrequency=('Frequency', 'median'))
segment_summary['CustomerShare'] = segment_summary['Customers'] / segment_summary['Customers'].sum()
segment_summary['RevenueShare'] = segment_summary['RevenueGBP'] / segment_summary['RevenueGBP'].sum()
segment_summary.sort_values('RevenueGBP', ascending=False)


## Interpretation and limitations

- November is the strongest full month in both years observed; Dec 2011 is partial and should not be compared as a full month.
- UK transactions dominate the recorded line sales. The top 10% of identified customers account for about 64% of their gross sales.
- The data contain missing customer IDs, special stock codes, cancellation/quantity mismatches, and a one-invoice item-volume outlier.
- There are no cost, margin, marketing-channel, or inventory data. Treat all recommendations as hypotheses to validate, not causal conclusions.

For the reproducible full set of CSVs and charts, run `python src/analyze_retail.py` from the project root.
